In [ ]:
import anndata
import pickle
from scipy.sparse import csc_matrix
import re
import numpy as np

In [ ]:
def str_to_day(s):
    vals = re.findall("[0-9]+", s)
    return int(np.mean([int(v) for v in vals]))

In [ ]:
raw_list = []
scvi_list = [] 

for stage_i, stage in enumerate(snakemake.params["stages"]):
    raw = anndata.read_h5ad(snakemake.input["raw"][stage_i])
    scvi2500_cellcycle = anndata.read_h5ad(snakemake.input["scvi"][stage_i])
    
    # normalize to CPM and set 0 counts
    scvi2500_cellcycle.X = (1000000 * scvi2500_cellcycle.X.T / scvi2500_cellcycle.X.sum(axis=1)).T
    scvi2500_cellcycle.X[scvi2500_cellcycle.X < 1] = 0
    scvi2500_cellcycle.X = csc_matrix(scvi2500_cellcycle.X)

    for adata in [raw, scvi2500_cellcycle]:
        if "anno_ordered" in adata.obs.columns:
            adata.obs["anno_time"] = adata.obs["anno_ordered"].apply(lambda s: f"day 01-05 {s[3:]}")  # cut the ordering
        else:
            adata.obs["anno_time"] = adata.obs["anno_og_time"].apply(lambda s: f"day {s[1:]}" if s != "undefined" else "day 01-42 unknown time")
        adata.obs["developmental_stage"] = stage
        adata.obs["anno_time"] = adata.obs.anno_time.apply(str_to_day)
        # adata.var["gene_name"] = adata.var.index  # doesn't work for some reason
    
    raw_list.append(raw)
    scvi_list.append(scvi2500_cellcycle)
    
    print(raw.shape)
    print(scvi2500_cellcycle.shape)

In [ ]:
# no idea, if we can trust this join operation tbh
raw = anndata.concat(raw_list, join="outer")
raw.var["gene_name"] = raw.var.index
scvi_pseudoraw = anndata.concat(scvi_list, join="outer")
scvi_pseudoraw.var["gene_name"] = scvi_pseudoraw.var.index

In [ ]:
scvi_pseudoraw.var

In [ ]:
import re







In [ ]:
raw.write_h5ad(snakemake.output.raw)
scvi_pseudoraw.write_h5ad(snakemake.output.scvi)